# EXP-039 | 4-Model Multi-seed (EXP-038 + LGB FE-v2+TE+ITE)

EXP-038(OOF 0.74090)에서 LGB(FE-v2+TE+ITE)를 추가해 모델 간 알고리즘 다양성 확보.

| 모델 | 피처셋 | 이유 |
|---|---|---|
| LGB | FE-v1 (85개) | 기존 최적화된 피처셋 |
| CAT | FE-v2+TE (98개) | TE로 범주형 강화 |
| XGB | FE-v2+TE+ITE (104개) | ITE로 상호작용 추가 |
| **LGB** | **FE-v2+TE+ITE (104개)** | **동일 피처, 다른 알고리즘 — XGB와 다양성** |

× 5 seeds = **20모델 Rank Average**

| 기준선 | EXP-038 OOF 0.74090 / 제출 미완 |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
N_FOLDS  = 5
SEEDS    = [42, 0, 123, 2024, 777]
EXP_NO   = 39
AUTHOR   = '조여진'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')
print(f'Seeds: {SEEDS}  /  N_FOLDS: {N_FOLDS}  →  총 {len(SEEDS)*4*N_FOLDS}회 학습  (4모델×5시드×5폴드)')

train: (256351, 69)  /  test: (90067, 68)
Seeds: [42, 0, 123, 2024, 777]  /  N_FOLDS: 5  →  총 100회 학습  (4모델×5시드×5폴드)


## 1. 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features_v1(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

def add_derived_features_v2(df):
    df = add_derived_features_v1(df)
    eps = 1e-6
    df['임신_성공률']   = df['총 임신 횟수'] / (df['총 시술 횟수'] + eps)
    df['시술_실패_횟수'] = (df['총 시술 횟수'] - df['총 임신 횟수']).clip(lower=0)
    egg_count = df['수집된 신선 난자 수'].fillna(-1)
    df['최적_난자수']    = ((egg_count >= 5) & (egg_count <= 15)).astype(int)
    df['나이_이식배아수'] = df['시술 당시 나이'] * df['이식된 배아 수'].fillna(0)
    return df

TE_COLS   = ['시술 시기 코드','시술 유형','특정 시술 유형','배란 유도 유형',
             '난자 출처','정자 출처','시술 당시 나이','총 시술 횟수','총 임신 횟수']
ITE_PAIRS = [('시술 당시 나이','시술 유형'),('시술 당시 나이','특정 시술 유형'),
             ('시술 당시 나이','난자 출처'),('시술 당시 나이','배란 유도 유형'),
             ('시술 유형','총 시술 횟수'),('난자 출처','시술 유형')]
TE_ALPHA, ITE_ALPHA = 10, 20

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_base_tr, X_base_te = preprocess(train_fe, test_fe)

X_v1_train = add_derived_features_v1(X_base_tr)
X_v1_test  = add_derived_features_v1(X_base_te)
X_v2_train = add_derived_features_v2(X_base_tr)
X_v2_test  = add_derived_features_v2(X_base_te)

TE_COLS   = [c for c in TE_COLS   if c in X_v2_train.columns]
ITE_PAIRS = [(c1,c2) for c1,c2 in ITE_PAIRS
             if c1 in X_v2_train.columns and c2 in X_v2_train.columns]

y_train = train[TARGET]
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'FE-v1: {X_v1_train.shape[1]}개  /  FE-v2: {X_v2_train.shape[1]}개')
print(f'TE: {len(TE_COLS)}개  /  ITE: {len(ITE_PAIRS)}쌍')

FE-v1: 85개  /  FE-v2: 89개
TE: 9개  /  ITE: 6쌍


## 2. 모델 파라미터

In [3]:
LGB_PARAMS_BASE = dict(
    objective='binary', metric='auc', verbosity=-1,
    num_threads=-1, is_unbalance=True, bagging_freq=1,
    learning_rate=0.035425966303284824,
    num_leaves=266, max_depth=5, min_child_samples=166,
    feature_fraction=0.5346439126449233,
    bagging_fraction=0.7122309235479091,
    lambda_l1=9.901034988600228,
    lambda_l2=2.213951873239442,
    min_gain_to_split=0.11418176854933762,
)
CAT_PARAMS_BASE = dict(
    iterations=2000, loss_function='Logloss', eval_metric='AUC',
    auto_class_weights='Balanced', thread_count=-1, verbose=False,
    early_stopping_rounds=100,
    learning_rate=0.018758723768855998, depth=6,
    l2_leaf_reg=9.189608434163782, min_data_in_leaf=19,
    subsample=0.8170921295501483, colsample_bylevel=0.6936810336930781,
)
XGB_PARAMS_BASE = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric='auc', tree_method='hist',
    n_jobs=-1, verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647, max_depth=4, min_child_weight=59,
    subsample=0.7663066457187595, colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928, reg_lambda=0.23932562420374562,
)

def rank_avg_normalize(arrays):
    ranks = np.stack([rankdata(a) for a in arrays], axis=1)
    avg = ranks.mean(axis=1)
    return (avg - avg.min()) / (avg.max() - avg.min())

print('파라미터 설정 완료')
print('모델 구성: LGB(FE-v1) + CAT(FE-v2+TE) + XGB(FE-v2+TE+ITE) + LGB(FE-v2+TE+ITE)')

파라미터 설정 완료
모델 구성: LGB(FE-v1) + CAT(FE-v2+TE) + XGB(FE-v2+TE+ITE) + LGB(FE-v2+TE+ITE)


## 3. Multi-seed 4-Model 학습

각 seed마다 4개 모델 학습. TE/ITE는 fold 내부에서 계산 (data leakage 방지).

In [4]:
n_train = len(X_v1_train)
n_test  = len(X_v1_test)

all_oof_lgb1 = []   # LGB FE-v1
all_oof_cat  = []   # CAT FE-v2+TE
all_oof_xgb  = []   # XGB FE-v2+TE+ITE
all_oof_lgb2 = []   # LGB FE-v2+TE+ITE  ← 신규
all_test_lgb1 = []
all_test_cat  = []
all_test_xgb  = []
all_test_lgb2 = []

for s_idx, seed in enumerate(SEEDS, 1):
    print(f'\n{"="*60}')
    print(f'Seed {seed}  ({s_idx}/{len(SEEDS)})')
    print(f'{"="*60}')

    lgb_p = {**LGB_PARAMS_BASE, 'seed': seed}
    cat_p = {**CAT_PARAMS_BASE, 'random_seed': seed}
    xgb_p = {**XGB_PARAMS_BASE, 'random_state': seed}
    skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    oof_lgb1 = np.zeros(n_train)
    oof_cat  = np.zeros(n_train)
    oof_xgb  = np.zeros(n_train)
    oof_lgb2 = np.zeros(n_train)
    test_lgb1 = np.zeros(n_test)
    test_cat  = np.zeros(n_test)
    test_xgb  = np.zeros(n_test)
    test_lgb2 = np.zeros(n_test)

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_v1_train, y_train), 1):
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        global_mean = y_tr.mean()

        # ── [1] LGB: FE-v1 ──────────────────────────────────
        X_tr_lgb1  = X_v1_train.iloc[tr_idx]
        X_val_lgb1 = X_v1_train.iloc[val_idx]
        ds_tr  = lgb.Dataset(X_tr_lgb1, label=y_tr)
        ds_val = lgb.Dataset(X_val_lgb1, label=y_val, reference=ds_tr)
        m = lgb.train(lgb_p, ds_tr, num_boost_round=2000, valid_sets=[ds_val],
                      callbacks=[lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(-1)])
        oof_lgb1[val_idx] = m.predict(X_val_lgb1)
        test_lgb1        += m.predict(X_v1_test) / N_FOLDS

        # ── TE/ITE 공통 계산 (fold 내부) ────────────────────
        X_tr_v2  = X_v2_train.iloc[tr_idx].copy()
        X_val_v2 = X_v2_train.iloc[val_idx].copy()
        X_te_v2  = X_v2_test.copy()
        for col in TE_COLS:
            grp = y_tr.groupby(X_tr_v2[col])
            sm  = (grp.count() * grp.mean() + TE_ALPHA * global_mean) / (grp.count() + TE_ALPHA)
            tc  = f'{col}_te'
            X_tr_v2[tc]  = X_tr_v2[col].map(sm).fillna(global_mean)
            X_val_v2[tc] = X_val_v2[col].map(sm).fillna(global_mean)
            X_te_v2[tc]  = X_te_v2[col].map(sm).fillna(global_mean)

        X_tr_ite  = X_tr_v2.copy()
        X_val_ite = X_val_v2.copy()
        X_te_ite  = X_te_v2.copy()
        for col1, col2 in ITE_PAIRS:
            key_tr  = X_tr_ite[col1].astype(str)  + '_' + X_tr_ite[col2].astype(str)
            key_val = X_val_ite[col1].astype(str) + '_' + X_val_ite[col2].astype(str)
            key_te  = X_te_ite[col1].astype(str)  + '_' + X_te_ite[col2].astype(str)
            grp = y_tr.groupby(key_tr)
            sm  = (grp.count() * grp.mean() + ITE_ALPHA * global_mean) / (grp.count() + ITE_ALPHA)
            ic  = f'{col1}_{col2}_ite'
            X_tr_ite[ic]  = key_tr.map(sm).fillna(global_mean)
            X_val_ite[ic] = key_val.map(sm).fillna(global_mean)
            X_te_ite[ic]  = key_te.map(sm).fillna(global_mean)

        # ── [2] CAT: FE-v2+TE ───────────────────────────────
        m = CatBoostClassifier(**cat_p)
        m.fit(X_tr_v2, y_tr, eval_set=Pool(X_val_v2, y_val), use_best_model=True)
        oof_cat[val_idx] = m.predict_proba(X_val_v2)[:, 1]
        test_cat        += m.predict_proba(X_te_v2)[:, 1] / N_FOLDS

        # ── [3] XGB: FE-v2+TE+ITE ───────────────────────────
        m = xgb.XGBClassifier(**xgb_p)
        m.fit(X_tr_ite, y_tr, eval_set=[(X_val_ite, y_val)], verbose=False)
        oof_xgb[val_idx] = m.predict_proba(X_val_ite)[:, 1]
        test_xgb        += m.predict_proba(X_te_ite)[:, 1] / N_FOLDS

        # ── [4] LGB: FE-v2+TE+ITE (신규) ────────────────────
        ds_tr2  = lgb.Dataset(X_tr_ite, label=y_tr)
        ds_val2 = lgb.Dataset(X_val_ite, label=y_val, reference=ds_tr2)
        m = lgb.train(lgb_p, ds_tr2, num_boost_round=2000, valid_sets=[ds_val2],
                      callbacks=[lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(-1)])
        oof_lgb2[val_idx] = m.predict(X_val_ite)
        test_lgb2        += m.predict(X_te_ite) / N_FOLDS

        f1 = roc_auc_score(y_val, oof_lgb1[val_idx])
        f2 = roc_auc_score(y_val, oof_cat[val_idx])
        f3 = roc_auc_score(y_val, oof_xgb[val_idx])
        f4 = roc_auc_score(y_val, oof_lgb2[val_idx])
        print(f'  Fold {fold}  LGB1={f1:.5f}  CAT={f2:.5f}  XGB={f3:.5f}  LGB2={f4:.5f}')

    seed_oof = rank_avg_normalize([oof_lgb1, oof_cat, oof_xgb, oof_lgb2])
    seed_auc = roc_auc_score(y_train, seed_oof)
    print(f'  → Seed {seed} OOF (4모델 Rank Avg): {seed_auc:.5f}')

    all_oof_lgb1.append(oof_lgb1)
    all_oof_cat.append(oof_cat)
    all_oof_xgb.append(oof_xgb)
    all_oof_lgb2.append(oof_lgb2)
    all_test_lgb1.append(test_lgb1)
    all_test_cat.append(test_cat)
    all_test_xgb.append(test_xgb)
    all_test_lgb2.append(test_lgb2)

print('\n학습 완료')


Seed 42  (1/5)
  Fold 1  LGB1=0.73812  CAT=0.73827  XGB=0.73833  LGB2=0.73819
  Fold 2  LGB1=0.74318  CAT=0.74338  XGB=0.74301  LGB2=0.74314
  Fold 3  LGB1=0.74072  CAT=0.74020  XGB=0.74036  LGB2=0.74046
  Fold 4  LGB1=0.73880  CAT=0.73864  XGB=0.73893  LGB2=0.73863
  Fold 5  LGB1=0.74160  CAT=0.74062  XGB=0.74156  LGB2=0.74136
  → Seed 42 OOF (4모델 Rank Avg): 0.74069

Seed 0  (2/5)
  Fold 1  LGB1=0.73952  CAT=0.73945  XGB=0.73938  LGB2=0.73958
  Fold 2  LGB1=0.74057  CAT=0.74013  XGB=0.74053  LGB2=0.74032
  Fold 3  LGB1=0.74097  CAT=0.74146  XGB=0.74109  LGB2=0.74120
  Fold 4  LGB1=0.73973  CAT=0.73962  XGB=0.73975  LGB2=0.73987
  Fold 5  LGB1=0.74002  CAT=0.74054  XGB=0.74108  LGB2=0.74067
  → Seed 0 OOF (4모델 Rank Avg): 0.74058

Seed 123  (3/5)
  Fold 1  LGB1=0.73881  CAT=0.73863  XGB=0.73877  LGB2=0.73911
  Fold 2  LGB1=0.74084  CAT=0.74039  XGB=0.74060  LGB2=0.74065
  Fold 3  LGB1=0.73981  CAT=0.73966  XGB=0.73986  LGB2=0.73969
  Fold 4  LGB1=0.74079  CAT=0.74011  XGB=0.74076  LGB2

## 4. Rank Average — OOF & Test

In [5]:
# 20개 모델 rank avg (5seeds × 4models)
all_oofs  = all_oof_lgb1  + all_oof_cat  + all_oof_xgb  + all_oof_lgb2
all_tests = all_test_lgb1 + all_test_cat + all_test_xgb + all_test_lgb2

oof_final  = rank_avg_normalize(all_oofs)
test_final = rank_avg_normalize(all_tests)

auc_final = roc_auc_score(y_train, oof_final)

BASELINE_032 = 0.74071
BASELINE_038 = 0.74090

print(f'[EXP-039] OOF AUC (20모델 Rank Avg): {auc_final:.5f}')
print(f'  vs EXP-032: {auc_final - BASELINE_032:+.5f}')
print(f'  vs EXP-038: {auc_final - BASELINE_038:+.5f}')
print(f'probability 범위: [{test_final.min():.4f}, {test_final.max():.4f}]')

# LGB2 단독 기여 확인
print('\n시드별 4모델 OOF AUC:')
for i, seed in enumerate(SEEDS):
    sa = roc_auc_score(y_train, rank_avg_normalize(
        [all_oof_lgb1[i], all_oof_cat[i], all_oof_xgb[i], all_oof_lgb2[i]]))
    sa3 = roc_auc_score(y_train, rank_avg_normalize(
        [all_oof_lgb1[i], all_oof_cat[i], all_oof_xgb[i]]))
    print(f'  Seed {seed:>4d}:  3모델={sa3:.5f}  4모델={sa:.5f}  (+{sa-sa3:.5f})')

[EXP-039] OOF AUC (20모델 Rank Avg): 0.74091
  vs EXP-032: +0.00020
  vs EXP-038: +0.00001
probability 범위: [0.0000, 1.0000]

시드별 4모델 OOF AUC:
  Seed   42:  3모델=0.74071  4모델=0.74069  (+-0.00001)
  Seed    0:  3모델=0.74056  4모델=0.74058  (+0.00002)
  Seed  123:  3모델=0.74056  4모델=0.74061  (+0.00005)
  Seed 2024:  3모델=0.74072  4모델=0.74070  (+-0.00002)
  Seed  777:  3모델=0.74077  4모델=0.74081  (+0.00004)


## 5. Submission 저장 & 실험 기록

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

auc_str   = f'{auc_final:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub_out   = sub.copy()
sub_out['probability'] = test_final
sub_out.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {out_fname}')
print(f'probability 범위: [{test_final.min():.4f}, {test_final.max():.4f}]')

저장: submission_exp039_조여진_07409.csv
probability 범위: [0.0000, 1.0000]


In [7]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin   = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill   = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font   = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_binary = (oof_final >= 0.5).astype(int)
params_str = (f'MultiSeed(5) × 4모델 | seeds={SEEDS} | '
              f'LGB(FE-v1)+CAT(FE-v2+TE)+XGB(FE-v2+TE+ITE)+LGB(FE-v2+TE+ITE) | 20모델 RankAvg')
NOTES    = ('EXP-038(15모델)에 LGB(FE-v2+TE+ITE) 추가 → 20모델 Rank Average. '
            'XGB와 동일 피처에 LGB 사용으로 알고리즘 다양성 확보. '
            'TE/ITE는 fold 내부 계산 (data leakage 없음).')
INSIGHTS = (f'EXP-038 OOF={BASELINE_038:.5f} 대비 {auc_final-BASELINE_038:+.5f} | '
            f'EXP-032 제출 0.74219 대비 | '
            f'20모델(5seeds×4models) RankAvg')

log_to_leaderboard(
    EXP_NO, AUTHOR, '4Model-MultiSeed(LGB×2+CAT+XGB×5seeds)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    auc_final, f'Stratified {N_FOLDS}-Fold × {len(SEEDS)} seeds',
    'mixed(v1/v2+TE+ITE)', X_v1_train.shape[1],
    'is_unbalance / Balanced / scale_pos_weight',
    'N', None, f'notebooks/35_multi_seed_4model_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-039 기록 완료 (row 36)
